# Ozon Multimodal Counterfeit Pipeline: Gated ConvNeXt E2

This notebook is a copy of the baseline pipeline for a gated-fusion E2 experiment with ConvNeXt Tiny image embeddings. The original notebook is intentionally left unchanged.

Sections:
1. Configuration
2. Optional dependency install
3. Imports
4. Manifest and split
5. Data checks
6. Training helpers
7. E0, E1, E2 experiments
8. Ablation table
9. Optional external inference


In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()

TRAIN_CSV = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_train.csv'
TEST_CSV = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_test.csv'
TRAIN_IMAGES_DIR = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_train_images'
TEST_IMAGES_DIR = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_test_images'

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
MANIFEST_PATH = ARTIFACTS_DIR / 'manifests/train_manifest.csv'
MANIFEST_SUMMARY_PATH = ARTIFACTS_DIR / 'manifests/train_manifest_summary.json'
DATA_CHECKS_PATH = ARTIFACTS_DIR / 'reports/data_checks.json'
ABLATION_PATH = ARTIFACTS_DIR / 'reports/ablation_table.csv'
EXTERNAL_PREDICTIONS_PATH = ARTIFACTS_DIR / 'predictions/external_test_predictions.csv'

INSTALL_DEPENDENCIES = False
RUN_E0 = False
RUN_E1 = False
RUN_E2 = True
RUN_EXTERNAL_INFERENCE = False
EXTERNAL_INFERENCE_EXPERIMENT = 'e2_gated_convnext'
E2_EXPERIMENT_NAME = 'e2_gated_convnext'
IMAGE_EXPERIMENT_NAMES = {E2_EXPERIMENT_NAME}

# Use a small integer for smoke runs on local hardware. Leave as None for the full dataset.
MAX_SAMPLES_PER_SPLIT = None

TEXT_MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
IMAGE_MODEL_NAME = 'convnext_tiny.fb_in22k_ft_in1k'
IMAGE_MODEL_CANDIDATES = (
    IMAGE_MODEL_NAME,
    'convnext_tiny.in12k_ft_in1k',
    'convnext_tiny',
)
IMAGE_EMBED_DIM_FALLBACK = 1280
DEVICE = 'mps'

EPOCHS = 12
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3
PROJECTION_DIM = 256
HIDDEN_DIM = 256
DROPOUT = 0.2
NUM_WORKERS = 0

PROJECT_ROOT


PosixPath('/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база')

In [2]:
if INSTALL_DEPENDENCIES:
    %pip install -r requirements.txt


In [3]:
import csv
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except ImportError:  # tqdm is optional for display only.
    tqdm = None

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ozon_counterfeit.constants import DEFAULT_SPLIT_RATIOS, RANDOM_SEED
from ozon_counterfeit.data import (
    build_external_test_manifest,
    build_train_manifest,
    load_external_test_frame,
    load_train_frame,
    merge_manifest_with_frame,
)
from ozon_counterfeit.encoders import (
    FrozenImageEncoder,
    FrozenTextEncoder,
    load_cached_array,
    save_cached_array,
)
from ozon_counterfeit.features import (
    TabularPreprocessor,
    build_modality_flags,
    build_text_corpus,
    extract_labels,
    infer_log_columns,
    infer_numeric_columns,
)
from ozon_counterfeit.splitting import compute_split_summary, greedy_group_split
from ozon_counterfeit.training import (
    NeuralTrainingConfig,
    load_pickle,
    save_pickle,
    choose_threshold_by_f1,
    compute_metrics,
    train_random_forest,
)
from ozon_counterfeit.utils import ensure_dir, load_json, resolve_torch_device, save_json, set_seed, slugify

set_seed(RANDOM_SEED)
print('Imports ready')


Imports ready


## Build Manifest And Seller-Aware Split


In [4]:
train_frame = load_train_frame(TRAIN_CSV)
split_map = greedy_group_split(train_frame, random_state=RANDOM_SEED)
manifest = build_train_manifest(train_frame, TRAIN_IMAGES_DIR, split_map)

ensure_dir(MANIFEST_PATH.parent)
manifest.to_csv(MANIFEST_PATH, index=False)

manifest_summary = compute_split_summary(manifest)
summary_payload = {
    'rows_total': int(len(manifest)),
    'rows_with_image': int(manifest['has_image'].sum()),
    'rows_without_image': int((1 - manifest['has_image']).sum()),
    'split_summary': manifest_summary.to_dict(orient='records'),
}
save_json(MANIFEST_SUMMARY_PATH, summary_payload)

display(manifest_summary)
print(f'Manifest saved to: {MANIFEST_PATH}')
print(f"Image coverage: {int(manifest['has_image'].sum())}/{len(manifest)}")


,split,rows,positives,negatives,positive_rate,unique_sellers,with_image_rows
0,test,32614,1958,30656,0.060036,60,32563
1,train,132068,9136,122932,0.069176,11073,132014
2,val,32516,1958,30558,0.060217,34,31883


Manifest saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/manifests/train_manifest.csv
Image coverage: 196460/197198


## Data Checks


In [5]:
manifest_from_disk = pd.read_csv(MANIFEST_PATH, dtype={'ItemID': str, 'SellerID': str})

manifest_full = train_frame[['row_id', 'ItemID', 'SellerID', 'resolution']].merge(
    manifest_from_disk,
    on=['row_id', 'ItemID', 'SellerID', 'resolution'],
    how='inner',
)

exact_mapping_ok = bool(
    manifest_full.loc[manifest_full['has_image'] == 1, 'image_path']
    .map(lambda value: Path(value).stem)
    .eq(manifest_full.loc[manifest_full['has_image'] == 1, 'ItemID'])
    .all()
)

split_seller_sets = {
    split_name: set(split_frame['SellerID'].astype(str))
    for split_name, split_frame in manifest_full.groupby('split')
}
seller_overlap_counts = {}
split_names = sorted(split_seller_sets)
for index, left_name in enumerate(split_names):
    for right_name in split_names[index + 1:]:
        overlap = split_seller_sets[left_name] & split_seller_sets[right_name]
        seller_overlap_counts[f'{left_name}__{right_name}'] = len(overlap)

data_checks = {
    'manifest_rows': int(len(manifest_full)),
    'train_rows': int(len(train_frame)),
    'rows_with_image': int(manifest_full['has_image'].sum()),
    'rows_without_image': int((1 - manifest_full['has_image']).sum()),
    'exact_mapping_ok': exact_mapping_ok,
    'seller_overlap_counts': seller_overlap_counts,
    'split_summary': manifest_summary.to_dict(orient='records'),
}
save_json(DATA_CHECKS_PATH, data_checks)

display(pd.DataFrame(manifest_summary))
data_checks


,split,rows,positives,negatives,positive_rate,unique_sellers,with_image_rows
0,test,32614,1958,30656,0.060036,60,32563
1,train,132068,9136,122932,0.069176,11073,132014
2,val,32516,1958,30558,0.060217,34,31883


{'manifest_rows': 197198,
 'train_rows': 197198,
 'rows_with_image': 196460,
 'rows_without_image': 738,
 'exact_mapping_ok': True,
 'seller_overlap_counts': {'test__train': 0, 'test__val': 0, 'train__val': 0},
 'split_summary': [{'split': 'test',
   'rows': 32614,
   'positives': 1958,
   'negatives': 30656,
   'positive_rate': 0.060035567547678914,
   'unique_sellers': 60,
   'with_image_rows': 32563},
  {'split': 'train',
   'rows': 132068,
   'positives': 9136,
   'negatives': 122932,
   'positive_rate': 0.06917648484114244,
   'unique_sellers': 11073,
   'with_image_rows': 132014},
  {'split': 'val',
   'rows': 32516,
   'positives': 1958,
   'negatives': 30558,
   'positive_rate': 0.060216508795669824,
   'unique_sellers': 34,
   'with_image_rows': 31883}]}

## Training Helpers


In [6]:
def sample_per_split(frame: pd.DataFrame, limit: int) -> pd.DataFrame:
    sampled = []
    for _, split_frame in frame.groupby('split', sort=False):
        take = min(len(split_frame), limit)
        sampled.append(split_frame.sample(n=take, random_state=RANDOM_SEED))
    return pd.concat(sampled, axis=0).sort_values(['split', 'row_id']).reset_index(drop=True)


class NotebookProjectionBlock(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)


class GatedFusionClassifier(nn.Module):
    def __init__(
        self,
        tabular_dim: int,
        text_dim: int,
        image_dim: int,
        projection_dim: int = 256,
        hidden_dim: int = 256,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.tabular_projector = NotebookProjectionBlock(tabular_dim, projection_dim, dropout)
        self.text_projector = NotebookProjectionBlock(text_dim, projection_dim, dropout)
        self.image_projector = NotebookProjectionBlock(image_dim, projection_dim, dropout)
        gate_input_dim = projection_dim * 3 + 3
        self.image_gate = nn.Sequential(
            nn.Linear(gate_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, projection_dim),
            nn.Sigmoid(),
        )
        fused_dim = projection_dim * 3 + 3
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        tabular_features: torch.Tensor,
        text_features: torch.Tensor,
        image_features: torch.Tensor,
        modality_flags: torch.Tensor,
    ) -> torch.Tensor:
        tabular_projection = self.tabular_projector(tabular_features)
        text_projection = self.text_projector(text_features)
        image_projection = self.image_projector(image_features)
        gate_inputs = torch.cat([tabular_projection, text_projection, image_projection, modality_flags], dim=1)
        gated_image_projection = image_projection * self.image_gate(gate_inputs)
        fused = torch.cat([tabular_projection, text_projection, gated_image_projection, modality_flags], dim=1)
        return self.classifier(fused).squeeze(1)


class NotebookFusionArrayDataset(Dataset):
    def __init__(
        self,
        tabular: np.ndarray,
        text: np.ndarray,
        image: np.ndarray,
        modality_flags: np.ndarray,
        labels: np.ndarray,
    ) -> None:
        self.tabular = torch.from_numpy(tabular.astype(np.float32))
        self.text = torch.from_numpy(text.astype(np.float32))
        self.image = torch.from_numpy(image.astype(np.float32))
        self.modality_flags = torch.from_numpy(modality_flags.astype(np.float32))
        self.labels = torch.from_numpy(labels.astype(np.float32))

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        return (
            self.tabular[index],
            self.text[index],
            self.image[index],
            self.modality_flags[index],
            self.labels[index],
        )


def make_fusion_loader(
    tabular: np.ndarray,
    text: np.ndarray,
    image: np.ndarray,
    flags: np.ndarray,
    labels: np.ndarray,
    batch_size: int,
    num_workers: int,
    shuffle: bool,
) -> DataLoader:
    dataset = NotebookFusionArrayDataset(tabular, text, image, flags, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=False)


def predict_gated_scores(model: nn.Module, loader: DataLoader, device: str, desc: str = 'Predicting') -> np.ndarray:
    scores = []
    model.eval()
    batch_iterator = tqdm(loader, desc=desc, leave=False) if tqdm is not None else loader
    with torch.no_grad():
        for tabular, text, image, flags, _ in batch_iterator:
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(scores, axis=0)


def train_gated_fusion(
    train_inputs: dict[str, np.ndarray],
    val_inputs: dict[str, np.ndarray],
    test_inputs: dict[str, np.ndarray],
    output_dir: Path,
    device: str,
    config: NeuralTrainingConfig,
) -> dict[str, object]:
    set_seed(config.random_state)
    output_dir = ensure_dir(output_dir)

    train_loader = make_fusion_loader(
        train_inputs['tabular'], train_inputs['text'], train_inputs['image'], train_inputs['flags'], train_inputs['labels'],
        config.batch_size, config.num_workers, shuffle=True,
    )
    val_loader = make_fusion_loader(
        val_inputs['tabular'], val_inputs['text'], val_inputs['image'], val_inputs['flags'], val_inputs['labels'],
        config.batch_size, config.num_workers, shuffle=False,
    )
    test_loader = make_fusion_loader(
        test_inputs['tabular'], test_inputs['text'], test_inputs['image'], test_inputs['flags'], test_inputs['labels'],
        config.batch_size, config.num_workers, shuffle=False,
    )

    model = GatedFusionClassifier(
        tabular_dim=train_inputs['tabular'].shape[1],
        text_dim=train_inputs['text'].shape[1],
        image_dim=train_inputs['image'].shape[1],
        projection_dim=config.projection_dim,
        hidden_dim=config.hidden_dim,
        dropout=config.dropout,
    ).to(device)

    positive_count = float(train_inputs['labels'].sum())
    negative_count = float(len(train_inputs['labels']) - positive_count)
    pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    history = []
    best_pr_auc = float('-inf')
    best_state = None
    best_threshold = 0.5
    epochs_without_improvement = 0

    epoch_iterator = range(1, config.epochs + 1)
    if tqdm is not None:
        epoch_iterator = tqdm(epoch_iterator, desc='Gated fusion epochs')

    for epoch in epoch_iterator:
        model.train()
        running_loss = 0.0
        train_batch_iterator = train_loader
        if tqdm is not None:
            train_batch_iterator = tqdm(train_loader, desc=f'Epoch {epoch}/{config.epochs}', leave=False)

        for tabular, text, image, flags, labels in train_batch_iterator:
            optimizer.zero_grad()
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            loss = criterion(logits, labels.to(device))
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * len(labels)
            if tqdm is not None:
                train_batch_iterator.set_postfix(loss=f'{float(loss.item()):.4f}')

        train_loss = running_loss / max(len(train_inputs['labels']), 1)
        val_scores = predict_gated_scores(model, val_loader, device, desc=f'Validation {epoch}/{config.epochs}')
        val_pr_auc = float(average_precision_score(val_inputs['labels'], val_scores))
        val_threshold = choose_threshold_by_f1(val_inputs['labels'], val_scores)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_pr_auc': val_pr_auc})

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
            best_threshold = val_threshold
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if tqdm is not None:
            epoch_iterator.set_postfix(
                train_loss=f'{train_loss:.4f}',
                val_pr_auc=f'{val_pr_auc:.4f}',
                best_pr_auc=f'{best_pr_auc:.4f}',
            )

        if epochs_without_improvement >= config.patience:
            break

    if best_state is None:
        raise RuntimeError('Training did not produce a best checkpoint.')

    model.load_state_dict(best_state)
    val_scores = predict_gated_scores(model, val_loader, device, desc='Final validation')
    test_scores = predict_gated_scores(model, test_loader, device, desc='Final test')
    metrics = {
        'validation': compute_metrics(val_inputs['labels'], val_scores, best_threshold),
        'test': compute_metrics(test_inputs['labels'], test_scores, best_threshold),
    }

    torch.save(model.state_dict(), output_dir / 'model.pt')
    save_json(output_dir / 'metrics.json', metrics)
    with (output_dir / 'training_history.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['epoch', 'train_loss', 'val_pr_auc'])
        writer.writeheader()
        writer.writerows(history)

    return {'model': model, 'threshold': best_threshold, 'metrics': metrics, 'history': history}


def make_image_encoder(device: str):
    errors = []
    for model_name in IMAGE_MODEL_CANDIDATES:
        try:
            return FrozenImageEncoder(model_name, device=device), model_name
        except Exception as error:
            errors.append(f'{model_name}: {error}')
    raise RuntimeError('Could not initialize any ConvNeXt image encoder candidates:\n' + '\n'.join(errors))


def maybe_encode_text(split_frame: pd.DataFrame, cache_path: Path, encoder, batch_size: int, force: bool = False) -> np.ndarray:
    cached = None if force else load_cached_array(cache_path)
    if cached is not None:
        return cached
    texts = build_text_corpus(split_frame)
    embeddings = encoder.encode(texts, batch_size=batch_size)
    return save_cached_array(cache_path, embeddings)


def maybe_encode_images(
    split_frame: pd.DataFrame,
    cache_path: Path,
    encoder,
    batch_size: int,
    num_workers: int,
    force: bool = False,
) -> np.ndarray:
    cached = None if force else load_cached_array(cache_path)
    if cached is not None:
        return cached
    image_paths = split_frame['image_path'].astype(str).tolist()
    embeddings = encoder.encode(
        image_paths,
        batch_size=batch_size,
        num_workers=num_workers,
    )
    return save_cached_array(cache_path, embeddings)


def save_metadata(
    output_dir: Path,
    experiment: str,
    numeric_columns: list[str],
    log_columns: list[str],
    train_split_frame: pd.DataFrame,
    eval_split_frame: pd.DataFrame,
    image_dim: int,
    text_dim: int,
    tabular_dim: int,
    image_model_name: str | None = None,
) -> None:
    payload = {
        'experiment': experiment,
        'train_csv': str(TRAIN_CSV),
        'manifest_path': str(MANIFEST_PATH),
        'text_model_name': TEXT_MODEL_NAME,
        'image_model_name': image_model_name or IMAGE_MODEL_NAME,
        'image_embedding_dim': image_dim,
        'text_embedding_dim': text_dim,
        'tabular_feature_dim': tabular_dim,
        'numeric_columns': numeric_columns,
        'log_columns': log_columns,
        'train_rows': int(len(train_split_frame)),
        'headline_eval_rows': int(len(eval_split_frame)),
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'projection_dim': PROJECTION_DIM,
        'hidden_dim': HIDDEN_DIM,
        'dropout': DROPOUT,
    }
    save_json(output_dir / 'metadata.json', payload)


def run_experiment(experiment: str, include_missing_images: bool = False):
    output_dir = ensure_dir(ARTIFACTS_DIR / 'experiments' / experiment)
    cache_dir = ensure_dir(output_dir / 'cache')

    dataset = merge_manifest_with_frame(train_frame, manifest)
    if MAX_SAMPLES_PER_SPLIT:
        dataset = sample_per_split(dataset, MAX_SAMPLES_PER_SPLIT)

    if experiment in IMAGE_EXPERIMENT_NAMES or not include_missing_images:
        dataset = dataset.loc[dataset['has_image'] == 1].copy()

    split_frames = {
        split_name: split_frame.reset_index(drop=True)
        for split_name, split_frame in dataset.groupby('split', sort=False)
    }
    for split_name in ('train', 'val', 'test'):
        if split_name not in split_frames:
            raise ValueError(f'Missing split: {split_name}')

    numeric_columns = infer_numeric_columns(train_frame)
    log_columns = infer_log_columns(numeric_columns)
    preprocessor = TabularPreprocessor(numeric_columns=numeric_columns, log_columns=log_columns)

    X_train_tab = preprocessor.fit_transform(split_frames['train'])
    X_val_tab = preprocessor.transform(split_frames['val'])
    X_test_tab = preprocessor.transform(split_frames['test'])
    y_train = extract_labels(split_frames['train'])
    y_val = extract_labels(split_frames['val'])
    y_test = extract_labels(split_frames['test'])

    save_pickle(output_dir / 'tabular_preprocessor.pkl', preprocessor)

    if experiment == 'e0':
        results = train_random_forest(
            X_train_tab,
            y_train,
            X_val_tab,
            y_val,
            X_test_tab,
            y_test,
            feature_names=preprocessor.get_feature_names(),
            output_dir=output_dir,
            random_state=RANDOM_SEED,
        )
        save_metadata(
            output_dir,
            experiment,
            numeric_columns,
            log_columns,
            split_frames['train'],
            split_frames['test'],
            image_dim=0,
            text_dim=0,
            tabular_dim=X_train_tab.shape[1],
        )
        return results

    text_encoder = FrozenTextEncoder(TEXT_MODEL_NAME, device=DEVICE)
    text_train = maybe_encode_text(
        split_frames['train'],
        cache_dir / f'text_train_{slugify(TEXT_MODEL_NAME)}.npy',
        text_encoder,
        batch_size=BATCH_SIZE,
    )
    text_val = maybe_encode_text(
        split_frames['val'],
        cache_dir / f'text_val_{slugify(TEXT_MODEL_NAME)}.npy',
        text_encoder,
        batch_size=BATCH_SIZE,
    )
    text_test = maybe_encode_text(
        split_frames['test'],
        cache_dir / f'text_test_{slugify(TEXT_MODEL_NAME)}.npy',
        text_encoder,
        batch_size=BATCH_SIZE,
    )

    metadata_image_model_name = IMAGE_MODEL_NAME

    if experiment in IMAGE_EXPERIMENT_NAMES:
        image_encoder, metadata_image_model_name = make_image_encoder(DEVICE)
        image_train = maybe_encode_images(
            split_frames['train'],
            cache_dir / f'image_train_{slugify(metadata_image_model_name)}.npy',
            image_encoder,
            batch_size=max(8, BATCH_SIZE // 4),
            num_workers=NUM_WORKERS,
        )
        image_val = maybe_encode_images(
            split_frames['val'],
            cache_dir / f'image_val_{slugify(metadata_image_model_name)}.npy',
            image_encoder,
            batch_size=max(8, BATCH_SIZE // 4),
            num_workers=NUM_WORKERS,
        )
        image_test = maybe_encode_images(
            split_frames['test'],
            cache_dir / f'image_test_{slugify(metadata_image_model_name)}.npy',
            image_encoder,
            batch_size=max(8, BATCH_SIZE // 4),
            num_workers=NUM_WORKERS,
        )
        image_dim = image_train.shape[1]
        train_flags = build_modality_flags(len(split_frames['train']), np.ones(len(split_frames['train']), dtype=np.float32))
        val_flags = build_modality_flags(len(split_frames['val']), np.ones(len(split_frames['val']), dtype=np.float32))
        test_flags = build_modality_flags(len(split_frames['test']), np.ones(len(split_frames['test']), dtype=np.float32))
    else:
        image_dim = IMAGE_EMBED_DIM_FALLBACK
        image_train = np.zeros((len(split_frames['train']), image_dim), dtype=np.float32)
        image_val = np.zeros((len(split_frames['val']), image_dim), dtype=np.float32)
        image_test = np.zeros((len(split_frames['test']), image_dim), dtype=np.float32)
        train_flags = build_modality_flags(len(split_frames['train']), np.zeros(len(split_frames['train']), dtype=np.float32))
        val_flags = build_modality_flags(len(split_frames['val']), np.zeros(len(split_frames['val']), dtype=np.float32))
        test_flags = build_modality_flags(len(split_frames['test']), np.zeros(len(split_frames['test']), dtype=np.float32))

    device = resolve_torch_device(DEVICE)
    config = NeuralTrainingConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        projection_dim=PROJECTION_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        random_state=RANDOM_SEED,
        num_workers=NUM_WORKERS,
    )

    results = train_gated_fusion(
        train_inputs={
            'tabular': X_train_tab,
            'text': text_train,
            'image': image_train,
            'flags': train_flags,
            'labels': y_train,
        },
        val_inputs={
            'tabular': X_val_tab,
            'text': text_val,
            'image': image_val,
            'flags': val_flags,
            'labels': y_val,
        },
        test_inputs={
            'tabular': X_test_tab,
            'text': text_test,
            'image': image_test,
            'flags': test_flags,
            'labels': y_test,
        },
        output_dir=output_dir,
        device=device,
        config=config,
    )

    save_metadata(
        output_dir,
        experiment,
        numeric_columns,
        log_columns,
        split_frames['train'],
        split_frames['test'],
        image_dim=image_dim,
        text_dim=text_train.shape[1],
        tabular_dim=X_train_tab.shape[1],
        image_model_name=metadata_image_model_name,
    )
    return results


## E0 Tabular Baseline


In [7]:
e0_results = None
if RUN_E0:
    e0_results = run_experiment('e0')
    display(pd.DataFrame([e0_results['metrics']['test']]).drop(columns=['classification_report', 'confusion_matrix']))
else:
    print('RUN_E0 is disabled')


RUN_E0 is disabled


## E1 Text + Tab Early Fusion


In [8]:
e1_results = None
if RUN_E1:
    e1_results = run_experiment('e1')
    display(pd.DataFrame([e1_results['metrics']['test']]).drop(columns=['classification_report', 'confusion_matrix']))
else:
    print('RUN_E1 is disabled')


RUN_E1 is disabled


## E2 Gated ConvNeXt Image + Text + Tab Fusion


In [10]:
e2_results = None
if RUN_E2:
    e2_results = run_experiment(E2_EXPERIMENT_NAME)
    display(pd.DataFrame([e2_results['metrics']['test']]).drop(columns=['classification_report', 'confusion_matrix']))
else:
    print('RUN_E2 is disabled')


/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/src/ozon_counterfeit/encoders.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Encoding images:   0%|          | 0/509 [00:00<?, ?it/s]

Gated fusion epochs:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 1/12:   0%|          | 0/516 [00:00<?, ?it/s]

Validation 1/12:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 2/12:   0%|          | 0/516 [00:00<?, ?it/s]

... (12 outputs left out)

Validation 8/12:   0%|          | 0/125 [00:00<?, ?it/s]

Final validation:   0%|          | 0/125 [00:00<?, ?it/s]

Final test:   0%|          | 0/128 [00:00<?, ?it/s]

,pr_auc,precision,recall,f1,threshold
0,0.539044,0.626145,0.523761,0.570395,0.759589


## Ablation Table


In [11]:
rows = []
for experiment in ('e0', 'e1', 'e2', E2_EXPERIMENT_NAME):
    experiment_dir = ARTIFACTS_DIR / 'experiments' / experiment
    metrics_path = experiment_dir / 'metrics.json'
    metadata_path = experiment_dir / 'metadata.json'
    if not metrics_path.exists() or not metadata_path.exists():
        continue

    metrics = load_json(metrics_path)
    metadata = load_json(metadata_path)
    rows.append(
        {
            'experiment': metadata['experiment'],
            'pr_auc': metrics['test']['pr_auc'],
            'precision': metrics['test']['precision'],
            'recall': metrics['test']['recall'],
            'f1': metrics['test']['f1'],
            'threshold': metrics['test']['threshold'],
            'headline_eval_rows': metadata['headline_eval_rows'],
        }
    )

ablation_table = pd.DataFrame(rows).sort_values('experiment').reset_index(drop=True)
ensure_dir(ABLATION_PATH.parent)
ablation_table.to_csv(ABLATION_PATH, index=False)
display(ablation_table)
print(f'Ablation table saved to: {ABLATION_PATH}')


,experiment,pr_auc,precision,recall,f1,threshold,headline_eval_rows
0,e0,0.224952,0.323248,0.311191,0.317105,0.102500,32563
1,e1,0.593553,0.572175,0.615738,0.593158,0.492610,32563
2,e2,0.606807,0.614045,0.540623,0.575000,0.623433,32563
3,e2_gated_convnext,0.539044,0.626145,0.523761,0.570395,0.759589,32563


Ablation table saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/reports/ablation_table.csv


## Optional External Test Inference

This cell does not use `ml_ozon_сounterfeit_test.csv` for evaluation. It only generates inference outputs.


In [ ]:
if RUN_EXTERNAL_INFERENCE:
    import torch

    experiment_dir = ARTIFACTS_DIR / 'experiments' / EXTERNAL_INFERENCE_EXPERIMENT
    metadata = load_json(experiment_dir / 'metadata.json')
    metrics = load_json(experiment_dir / 'metrics.json')
    threshold = float(metrics['validation']['threshold'])
    device = resolve_torch_device(DEVICE)

    test_source = load_external_test_frame(TEST_CSV)
    test_manifest = build_external_test_manifest(test_source, TEST_IMAGES_DIR)
    external_test_frame = merge_manifest_with_frame(test_source, test_manifest)
    preprocessor = load_pickle(experiment_dir / 'tabular_preprocessor.pkl')

    if metadata['experiment'] == 'e0':
        rf_model = load_pickle(experiment_dir / 'model.pkl')
        scores = rf_model.predict_proba(preprocessor.transform(external_test_frame))[:, 1]
    else:
        model = GatedFusionClassifier(
            tabular_dim=int(metadata['tabular_feature_dim']),
            text_dim=int(metadata['text_embedding_dim']),
            image_dim=int(metadata['image_embedding_dim']),
            projection_dim=int(metadata['projection_dim']),
            hidden_dim=int(metadata['hidden_dim']),
            dropout=float(metadata['dropout']),
        )
        state_dict = torch.load(experiment_dir / 'model.pt', map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        model.to(device)

        tabular = preprocessor.transform(external_test_frame)
        text_encoder = FrozenTextEncoder(str(metadata['text_model_name']), device=device)
        text = text_encoder.encode(build_text_corpus(external_test_frame), batch_size=BATCH_SIZE)

        if metadata['experiment'] in IMAGE_EXPERIMENT_NAMES:
            image_encoder = FrozenImageEncoder(str(metadata['image_model_name']), device=device)
            image = np.zeros((len(external_test_frame), int(metadata['image_embedding_dim'])), dtype=np.float32)
            has_image_mask = external_test_frame['has_image'].to_numpy(dtype=bool)
            if has_image_mask.any():
                encoded = image_encoder.encode(
                    external_test_frame.loc[has_image_mask, 'image_path'].astype(str).tolist(),
                    batch_size=max(8, BATCH_SIZE // 4),
                    num_workers=NUM_WORKERS,
                )
                image[has_image_mask] = encoded
            flags = build_modality_flags(len(external_test_frame), external_test_frame['has_image'].to_numpy(dtype=np.float32))
        else:
            image = np.zeros((len(external_test_frame), int(metadata['image_embedding_dim'])), dtype=np.float32)
            flags = build_modality_flags(len(external_test_frame), np.zeros(len(external_test_frame), dtype=np.float32))

        with torch.no_grad():
            logits = model(
                torch.from_numpy(tabular).to(device),
                torch.from_numpy(text).to(device),
                torch.from_numpy(image).to(device),
                torch.from_numpy(flags).to(device),
            )
            scores = torch.sigmoid(logits).cpu().numpy()

    predictions = (scores >= threshold).astype(int)
    external_predictions = pd.DataFrame(
        {
            'id': external_test_frame['id'],
            'ItemID': external_test_frame['ItemID'],
            'score': scores,
            'prediction': predictions,
            'has_image': external_test_frame['has_image'],
        }
    )

    ensure_dir(EXTERNAL_PREDICTIONS_PATH.parent)
    external_predictions.to_csv(EXTERNAL_PREDICTIONS_PATH, index=False)
    display(external_predictions.head())
    print(f'External predictions saved to: {EXTERNAL_PREDICTIONS_PATH}')
else:
    print('RUN_EXTERNAL_INFERENCE is disabled')
